# Step 1.1 — Data Sourcing

**Goal:** Load all **currently active** laws from the *Systematische Gesetzessammlung Basel-Stadt (SGBS)* via the REST API on [data.bs.ch](https://data.bs.ch) (Dataset ID: `100354`) and persist them locally as a CSV file.

**Why this approach?**  
The dataset is publicly available and machine-readable via a REST API — no manual download or labelling required. We filter for active laws only (`is_active=True`), because repealed laws are irrelevant to our research question about the *current* semantic structure of Basel-Stadt legislation. Instead of paginating through records (which hits a hard API limit of 10'000), we use the dedicated **export endpoint**, which returns the full filtered dataset as a single CSV with no record cap. We persist the raw data to `data/raw/` so subsequent steps work offline and reproducibly.

## 1. Setup

In [37]:
import requests
import pandas as pd
import io
import os

RAW_DATA_PATH = "../data/raw/sgbs_raw.csv"
os.makedirs("../data/raw", exist_ok=True)

## 2. API Configuration

We use the Opendatasoft **export endpoint** instead of the records endpoint. This bypasses the 10'000-record pagination limit and returns the full dataset as a CSV in a single request.

Filter: `is_active=True` — only laws currently in force.

In [40]:
EXPORT_URL = "https://data.bs.ch/api/explore/v2.1/catalog/datasets/100354/exports/csv"

params = {
    "delimiter":",",
    "use_labels": False
}

## 3. Fetch and Load Data

In [43]:
response = requests.get(EXPORT_URL, params=params)
response.raise_for_status()

df = pd.read_csv(io.StringIO(response.text))
df = df[df["is_active"] == True]
print(f"Active records after filter: {len(df)}")
print(f"Total active records loaded: {len(df)}")
print(f"\nColumns:\n{df.columns.tolist()}")

Active records after filter: 908
Total active records loaded: 908

Columns:
['index', 'parent', 'children', 'identifier', 'title', 'identifier_full', 'title_full', 'id', 'systematic_number', 'is_active', 'v_id', 'title_de', 'keywords_de', 'info_badge', 'info_badge_date', 'type', 'continuation_type', 'previous_type', 'version_active_since', 'family_active_since', 'version_inactive_since', 'version_found_at', 'category_id', 'category_name', 'original_url_de', 'url_de', 'text_of_law', 'gesetzestext_html', 'version_url_de', 'tolsv_dtah_url', 'tolsv_dtah_file_size']


## 4. Inspect

In [42]:
print("Shape:", df.shape)
print("\nMissing values per column:")
print(df.isnull().sum())
df.head(3)

Shape: (10943, 31)

Missing values per column:
index                         1
parent                       15
children                  10854
identifier                    1
title                         1
identifier_full               1
title_full                    1
id                           74
systematic_number            74
is_active                    74
v_id                         74
title_de                     74
keywords_de                6093
info_badge                   74
info_badge_date           10056
type                         74
continuation_type           982
previous_type              1442
version_active_since         74
family_active_since          74
version_inactive_since     9734
version_found_at             74
category_id                  74
category_name                74
original_url_de            1626
url_de                       74
text_of_law                2956
gesetzestext_html          2956
version_url_de             2956
tolsv_dtah_url           

,index,parent,children,identifier,title,identifier_full,title_full,id,systematic_number,is_active,...,version_found_at,category_id,category_name,original_url_de,url_de,text_of_law,gesetzestext_html,version_url_de,tolsv_dtah_url,tolsv_dtah_file_size
0,1436.0,1434.0,NaN,RiE 64,Steuerordnung der Gemeinde Riehen,RiE/RiE 6/RiE 64,Einwohnergemeinde Riehen/Finanzen · Gemeindegr...,4815.0,RiE 640.100,False,...,2016-03-31,9.0,Gemeindeerlass,https://www.gesetzessammlung.bs.ch/data/RiE%20...,https://www.lexfind.ch/tol/4815/de,Steuerordnung | Riehen: Einwohnergemeinde\n\n\...,<h2 class='header_text'>\n Steuerordnung | Ri...,https://www.gesetzessammlung.bs.ch/app/de/text...,https://www.lexfind.ch/tolv/21101/de,138197.0
1,1436.0,1434.0,NaN,RiE 64,Steuerordnung der Gemeinde Riehen,RiE/RiE 6/RiE 64,Einwohnergemeinde Riehen/Finanzen · Gemeindegr...,4815.0,RiE 640.100,False,...,2015-05-20,9.0,Gemeindeerlass,https://www.gesetzessammlung.bs.ch/data/RiE%20...,https://www.lexfind.ch/tol/4815/de,Steuerordnung | Riehen: Einwohnergemeinde\n\n\...,<h2 class='header_text'>\n Steuerordnung | Ri...,https://www.gesetzessammlung.bs.ch/app/de/text...,https://www.lexfind.ch/tolv/21099/de,137292.0
2,1436.0,1434.0,NaN,RiE 64,Steuerordnung der Gemeinde Riehen,RiE/RiE 6/RiE 64,Einwohnergemeinde Riehen/Finanzen · Gemeindegr...,4815.0,RiE 640.100,False,...,2015-01-01,9.0,Gemeindeerlass,https://www.gesetzessammlung.bs.ch/data/RiE%20...,https://www.lexfind.ch/tol/4815/de,Steuerordnung | Riehen: Einwohnergemeinde\n\n\...,<h2 class='header_text'>\n Steuerordnung | Ri...,https://www.gesetzessammlung.bs.ch/app/de/text...,https://www.lexfind.ch/tolv/21098/de,81611.0


## 5. Select Relevant Columns

| Column | Purpose |
|--------|--------|
| `title_de` | Law title — human-readable label |
| `text_of_law` | Full legal text — primary NLP input |
| `systematic_number` | Official SGBS category — used later for **validation** (human logic vs. ML logic) |
| `category_name` | Human-readable category name |

In [44]:
COLS_TO_KEEP = ["title_de", "text_of_law", "systematic_number", "category_name"]

available_cols = [c for c in COLS_TO_KEEP if c in df.columns]
print(f"Keeping columns: {available_cols}")

df_slim = df[available_cols].copy()

before = len(df_slim)
df_slim = df_slim[df_slim["text_of_law"].notna() & (df_slim["text_of_law"].str.strip() != "")]
after = len(df_slim)
print(f"Dropped {before - after} rows with empty text. Remaining: {after}")

df_slim.head(3)

Keeping columns: ['title_de', 'text_of_law', 'systematic_number', 'category_name']
Dropped 8 rows with empty text. Remaining: 900


,title_de,text_of_law,systematic_number,category_name
8,Gesetz über die Handänderungssteuer,Handänderungssteuer: Gesetz | Spezielle Steuer...,650.100,Gesetz
17,Gesetz über die Besteuerung der Motorfahrzeuge,Besteuerung der Motorfahrzeuge: Gesetz | Spezi...,650.500,Gesetz
32,Ordnung für die Benützung von Lokalitäten im V...,Benützung der Gemeindelokalitäten: Ordnung | R...,RiE 681.300,Gemeindeerlass


## 6. Persist Raw Data

In [45]:
df_slim.to_csv(RAW_DATA_PATH, index=False)
print(f"Saved {len(df_slim)} records to {RAW_DATA_PATH}")

Saved 900 records to ../data/raw/sgbs_raw.csv


## 7. Summary

**What we did:**
- Filtered 10943 laws for active laws today: is_active=True
- Retained four columns: `title_de`, `text_of_law`, `systematic_number`, `category_name`
- Dropped records with empty `text_of_law`
- Persisted the result as `data/raw/sgbs_raw.csv`

**Next step:** Steps 1.2–1.4 — cleaning (HTML removal, stopwords, legal boilerplate) and lemmatization.